# Fetch Ingested Videos
Fetches recently (last 24 hours) ingested videos and prints 10 random videos with views.

In [ ]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm a successful connection
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"An error occurred: {e}")
    print("\nMake sure you have added 'MONGODB_URI' to your Colab Secrets (the key icon on the left).")

In [ ]:
from bson.objectid import ObjectId
import datetime
import random

# Predikto database and collection mapping per documentation (assumed generic Colab use case since this is an analytics notebook similar to Graphiko)
db = client['predikto']
videos_col = db['publicvideos']

# Fetch videos ingested in the last 24 hours using timezone-aware UTC datetime
twenty_four_hours_ago = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)
oid = ObjectId.from_datetime(twenty_four_hours_ago)
query = {'_id': {'$gte': oid}}

# Use aggregate to randomly sample 10 videos
pipeline = [
    {'$match': query},
    {'$sample': {'size': 10}}
]
recent_videos = list(videos_col.aggregate(pipeline))

if not recent_videos:
    print('No videos found in the last 24 hours.')
else:
    print(f'Found {len(recent_videos)} videos in the random sample. Printing titles and view counts:')
    for video in recent_videos:
        title = video.get('title', 'Unknown Title')
        views = video.get('viewCount', 0)
        print(f'- {title} | Views: {views}')


In [ ]:
# Install InfluxDB client and pandas
!pip install influxdb-client pandas

### Create 10k Training Dataset using InfluxDB
Queries `channel_metrics` iterating forwards day-by-day to find the true first snapshot (T=0) and the snapshot approximately 24 hours later (T=24) for each video without overloading the DB.

In [ ]:
import pandas as pd
from influxdb_client import InfluxDBClient
from google.colab import userdata
import datetime

try:
    # Retrieve InfluxDB connection details from Colab Secrets
    url = userdata.get('INFLUXDB_URL')
    token = userdata.get('INFLUXDB_TOKEN')
    org = userdata.get('INFLUXDB_ORG')
    bucket = userdata.get('INFLUXDB_BUCKET')

    client = InfluxDBClient(url=url, token=token, org=org)
    query_api = client.query_api()
    print("Successfully connected to InfluxDB!")
except Exception as e:
    print(f"An error occurred during InfluxDB connection: {e}")
    print("\nMake sure you have added INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, and INFLUXDB_BUCKET to your Colab Secrets.")

def format_flux_time(dt):
    """Formats timezone-aware UTC datetime for Flux queries."""
    return dt.isoformat().replace('+00:00', 'Z')

def query_day(start_time, end_time):
    """Queries a specific time window and returns a pivoted DataFrame."""
    flux_query = f'''
        from(bucket: "{bucket}")
            |> range(start: {format_flux_time(start_time)}, stop: {format_flux_time(end_time)})
            |> filter(fn: (r) => r["_measurement"] == "channel_metrics")
            |> filter(fn: (r) => r["_field"] == "views" or r["_field"] == "videoId")
            |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
            |> keep(columns: ["_time", "videoId", "views"])
    '''
    
    try:
        df = query_api.query_data_frame(flux_query)
        if isinstance(df, list):
            if not df:
                return pd.DataFrame()
            df = pd.concat(df, ignore_index=True)
        if not df.empty and '_time' in df.columns:
            df['_time'] = pd.to_datetime(df['_time'])
        return df
    except Exception as e:
        print(f"Error querying window {start_time} to {end_time}: {e}")
        return pd.DataFrame()


training_data = []
target_rows = 10000
days_to_search = 365 # Max historical lookback in days

# Start looking from 365 days ago, stepping forwards, so we get the true T0 of videos
start_time = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0) - datetime.timedelta(days=days_to_search)
end_time = start_time + datetime.timedelta(days=1)

print(f"Fetching data day by day (forwards) until {target_rows} rows are found...")

# Track videos we've seen: { videoId: {'time_t0': time, 'views_t0': views} }
first_seen_dict = {}
# Track videos already added to the training dataset to avoid duplicates
completed_videos = set()

try:
    for i in range(days_to_search):
        print(f"Searching window: {start_time.date()} to {end_time.date()}...", end=" ")
        df = query_day(start_time, end_time)
        
        if df.empty:
            print("No data found.")
        else:
            # Sort entirely by time first so we process chronologically within the day
            df = df.sort_values(by='_time')
            added_this_window = 0
            
            for _, row in df.iterrows():
                vid = row['videoId']
                if vid in completed_videos:
                    continue
                    
                curr_time = row['_time']
                curr_views = row['views']
                
                if vid not in first_seen_dict:
                    # First time seeing this video across all windows
                    first_seen_dict[vid] = {
                        'time_t0': curr_time,
                        'views_t0': curr_views
                    }
                else:
                    # We already have the T0 for this video, check if this is the 24h mark
                    t0_time = first_seen_dict[vid]['time_t0']
                    target_24h = t0_time + pd.Timedelta(hours=24)
                    
                    time_diff = abs(curr_time - target_24h)
                    
                    # If it's within a 4 hour tolerance window of the 24 hour mark
                    # (If multiple snapshots fall in this tolerance, the first one encountered chronologically is used)
                    if time_diff <= pd.Timedelta(hours=4):
                        training_data.append({
                            'videoId': vid,
                            'views_t0': first_seen_dict[vid]['views_t0'],
                            'views_t24': curr_views,
                            'time_t0': t0_time,
                            'time_t24': curr_time
                        })
                        completed_videos.add(vid)
                        added_this_window += 1
                        
                        if len(training_data) >= target_rows:
                            break
            
            print(f"Added {added_this_window} videos. Total so far: {len(training_data)}")
            
        if len(training_data) >= target_rows:
            print("Reached target 10k rows!")
            break
            
        # Move the window forward by one day
        start_time = end_time
        end_time = start_time + datetime.timedelta(days=1)

    training_dataset = pd.DataFrame(training_data)
    
    if not training_dataset.empty:
        print(f"\nCreated training dataset with {len(training_dataset)} rows.")
        print("\n--- Sample Dataset ---")
        print(training_dataset.head())
        
        sample_video_id = training_dataset.iloc[0]['videoId']
        print(f"\n--- Debugging Input for video {sample_video_id} ---")
        
        # Re-query all data for the sample video to show debug info
        debug_query = f'''
            from(bucket: "{bucket}")
                |> range(start: 0)
                |> filter(fn: (r) => r["_measurement"] == "channel_metrics")
                |> filter(fn: (r) => r["_field"] == "views" or r["_field"] == "videoId")
                |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
                |> filter(fn: (r) => r["videoId"] == "{sample_video_id}")
                |> keep(columns: ["_time", "views"])
        '''
        debug_df = query_api.query_data_frame(debug_query)
        if isinstance(debug_df, list):
             debug_df = pd.concat(debug_df, ignore_index=True)
        
        if not debug_df.empty:
             debug_df['_time'] = pd.to_datetime(debug_df['_time'])
             debug_df = debug_df.sort_values(by='_time')
             print(debug_df[['_time', 'views']])
    else:
        print("\nNo data could be found matching the criteria.")
        
except Exception as e:
    print(f"An error occurred during data processing: {e}")
